# Read dictionary values in the ERD payload

Follow one dictionary through the actual Maxitor query.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, BaseEnumEntity, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


class ShopDomain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currencies accepted by the shop", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the shop's currency vocabulary."""

    symbol: str = Field(description="Currency symbol")

    EUR = value(code="eur", display_name="Euro", symbol="€")
    USD = value(code="usd", display_name="US dollar", symbol="$")


@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Associate an amount with its currency."""

    id: str = Field(description="Tag identifier")
    amount: float = Field(description="Price amount")
    currency: CurrencyEnumEntity = Field(description="Money unit")

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
import json

from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.maxitor.model.diagrams.actions.list_entities_action import ListEntitiesAction
from aoa.maxitor.model.diagrams.resources.duckdb_graph_resource import DuckDBGraphResource

machine = ActionProductMachine(loggers=[])
domain = next(node.node_id for node in machine.graph_coordinator.get_all_nodes() if node.label == "ShopDomain")
store = DuckDBGraphResource.build_from_json(json.loads(machine.graph_coordinator.to_json()))
diagram = ListEntitiesAction._slice_payload(store, domain, include_neighbors=False)
currency = next(item for item in diagram["entities"] if item["label"] == "CurrencyEnumEntity")
tag = next(item for item in diagram["entities"] if item["label"] == "PriceTagCoreEntity")
print(currency["values"][0])
print(next(field["reference_qualname"] for field in tag["fields"] if field["name"] == "currency").split(".")[-1])

{'code': 'eur', 'display_name': 'Euro', 'properties': [{'name': 'symbol', 'value': '€'}]}
CurrencyEnumEntity


## What the result means

Maxitor reads the actual graph export and produces display rows for the currency card. The property object becomes a list of `{name, value}` text items, while the reference field keeps the dictionary’s qualified name. This is a presentation payload, not a change to the Python currency.